# Quick Start Guide
github link: [cytozip](https://github.com/DingWB/cytozip)

## 1. Installation

### conda
https://anaconda.org/bioconda/cytozip
```shell
conda install -c bioconda cytozip
# or
mamba install -c bioconda cytozip
```

### pip
```shell
# Prerequisites (one of):
#   conda install -c bioconda htslib libdeflate          # recommended
#   apt-get install libhts-dev libdeflate-dev            # Debian/Ubuntu
#   brew install htslib libdeflate                       # macOS
pip install cytozip
# or reinstall the latest version from github
pip uninstall -y cytozip && pip install git+http://github.com/DingWB/cytozip
```

In [1]:
# Make sure czip is installed:
!which czip

~/Software/conda/m3c/bin/czip


## 1. Build reference cz file from a reference genome fasta file
The reference holds the genome-wide (chrom, pos, strand, context) axis. Per-cell .cz then store only mc/cov and reuse the reference’s positions, cutting per-cell size by ~5×.

In [2]:
!time czip build_ref --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \ # reference genome 
    --output ~/Ref/hg38/hg38_with_chrL.allc.cz  \ # output reference cz file 
    --jobs 24 \ # number of CPU 
    --chrom_size ~/Ref/hg38/hg38_ucsc_with_chrL.main.chrom.sizes \ # restrited to main chromosomes (optional)

2026-07-22 11:34:57.492 | DEBUG    | cytozip.allc:WriteC:66 - chr1
2026-07-22 11:34:58.126 | DEBUG    | cytozip.allc:WriteC:66 - chr10
2026-07-22 11:34:58.860 | DEBUG    | cytozip.allc:WriteC:66 - chr11
2026-07-22 11:34:59.571 | DEBUG    | cytozip.allc:WriteC:66 - chr12
2026-07-22 11:35:00.213 | DEBUG    | cytozip.allc:WriteC:66 - chr13
2026-07-22 11:35:00.779 | DEBUG    | cytozip.allc:WriteC:66 - chr14
2026-07-22 11:35:01.325 | DEBUG    | cytozip.allc:WriteC:66 - chr15
2026-07-22 11:35:01.821 | DEBUG    | cytozip.allc:WriteC:66 - chr16
2026-07-22 11:35:02.279 | DEBUG    | cytozip.allc:WriteC:66 - chr17
2026-07-22 11:35:02.756 | DEBUG    | cytozip.allc:WriteC:66 - chr18
2026-07-22 11:35:03.066 | DEBUG    | cytozip.allc:WriteC:66 - chr19
2026-07-22 11:35:04.575 | DEBUG    | cytozip.allc:WriteC:66 - chr2
2026-07-22 11:35:04.794 | DEBUG    | cytozip.allc:WriteC:66 - chr20
2026-07-22 11:35:05.038 | DEBUG    | cytozip.allc:WriteC:66 - chr21
2026-07-22 11:35:05.320 | DEBUG    | cytozip.allc:

In [3]:
!czip header -I ~/Ref/hg38/hg38_with_chrL.allc.cz

magic  :  b'CZIP'
version  :  0.35
total_size  :  1430352768
message  :  /home/x-wding2/Ref/hg38/hg38_ucsc_with_chrL.fa
formats  :  ['Q', 'c', '3s']
columns  :  ['pos', 'strand', 'context']
sort_col  :  0
delta_cols  :  [0]
chunk_dims  :  ['chrom']
header_size  :  102


In [5]:
! czip view -I ~/Ref/hg38/hg38_with_chrL.allc.cz --show_dims 0 | head

chrom	pos	strand	context
chr1	10004	+	CCC
chr1	10005	+	CCT
chr1	10006	+	CTA
chr1	10010	+	CCC
chr1	10011	+	CCT
chr1	10012	+	CTA
chr1	10016	+	CCC
chr1	10017	+	CCT
chr1	10018	+	CTA


## 2. Call methylation from bam file and save as .cz file

### 2.1 Download example bam files

In [ ]:
```shell
# Download example bam files from figshare with your browser: https://figshare.com/ndownloader/files/63998524
# or download with command line using pyfigshare:
pip install pyfigshare # https://github.com/DingWB/pyfigshare
figshare download 32095567 --file_id 66902144,66902129 --outdir cytozip_example_data

### 2.2 Call DNA methylation using cytozip

In [ ]:
czip bam_to_cz --input cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam \
               --genome ~/Ref/hg38/hg38_ucsc_with_chrL.fa \
               --output cytozip_example_data/hg38_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz \
               

In [ ]:
# Python API
from cytozip.bam import bam_to_cz
bam_to_cz(bam_path="deduped_bam/UWA7648_CX1819_NAC_1_P10-1-K18-A10.bam", \
                  genome=str(REF_FA), \
                  output="UWA7648_CX1819_NAC_1_P10-1-K18-A10.cz", \
                  reference=str(REF_CZ))

## 1. Convert allc/bed/stdin to .cz

In [3]:
! czip tocz -h

usage: czip tocz [-h] -O OUTPUT [-I INPUT] [-F FORMATS] [-C COLUMNS]
                 [-D CHUNK_DIMS] [-u USECOLS] [-d KEY_COLS] [-s SEP]
                 [-c BATCH_SIZE] [--header HEADER] [--skiprows SKIPROWS]
                 [-m MESSAGE] [-l LEVEL] [--delta_cols DELTA_COLS]

options:
  -h, --help            show this help message and exit
  -O OUTPUT, --output OUTPUT
                        output .cz file (default: None)
  -I INPUT, --input INPUT
                        input file (stdin if omitted) (default: None)
  -F FORMATS, --formats FORMATS
                        column formats, comma-separated (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names, comma-separated (default: ['mc', 'cov'])
  -D CHUNK_DIMS, --chunk_dims CHUNK_DIMS
                        chunk-key (dimension) names, comma-separated (default:
                        ['chrom'])
  -u USECOLS, --usecols USECOLS
                        column indices to pack, comma-separated (de

In [5]:
! czip allc2cz -h

usage: czip allc2cz [-h] -I INPUT -O OUTFILE [-r REFERENCE]
                    [--missing-value MISSING_VALUE] [-F FORMATS] [-C COLUMNS]
                    [-D DIMENSIONS] [-u USECOLS] [--pr PR] [--pa PA] [-s SEP]
                    [--path-to-chrom PATH_TO_CHROM] [-c CHUNKSIZE]

options:
  -h, --help            show this help message and exit
  -I INPUT, --input INPUT
                        input allc.tsv.gz (default: None)
  -O OUTFILE, --outfile OUTFILE
                        output .cz file (default: None)
  -r REFERENCE, --reference REFERENCE
                        reference .cz file (default: None)
  --missing-value MISSING_VALUE
                        missing value fill (default: [0, 0])
  -F FORMATS, --formats FORMATS
                        column formats (default: ['B', 'B'])
  -C COLUMNS, --columns COLUMNS
                        column names (default: ['mc', 'cov'])
  -D DIMENSIONS, --dimensions DIMENSIONS
                        dimension names (default: ['chrom'])


## 2. Python API (Quick Overview)

In [ ]:
import cytozip as czip

# Read a .cz file using Python API
reader = czip.Reader("test1.cz")
reader.print_header()

In [ ]:
# Fetch records for a chunk (dimension)
for record in reader.fetch(("chr1",)):
    print(record)

In [ ]:
# Query by coordinate range
for record in reader.query(chunk_key="chr8", start=3002101, end=3002103, printout=False):
    print(record)
reader.close()

## 3. Remote Reading (from URL)
czip supports reading .cz files directly from HTTP/HTTPS URLs without downloading the entire file. This uses HTTP Range requests and a chunk index for O(1) lookup.

In [ ]:
import cytozip as czip

# Open a remote .cz file via URL
# url = "https://example.com/path/to/file.cz"
# reader = czip.Reader.from_url(url)
# reader.print_header()

# Or simply pass a URL to Reader (auto-detected)
# reader = czip.Reader(url)
# for record in reader.fetch(("chr1",)):
#     print(record)
# reader.close()
print("Remote reading requires a .cz file hosted on an HTTP server supporting Range requests.")